# Sonnet Generation with GPT-2

This notebook trains and evaluates the `SonnetGPT` model on Google Colab. It clones the repo, installs dependencies, runs training, and saves the generated sonnets.

## 1. Mount Google Drive & Clone Repo

In [7]:
import os

REPO_URL = 'https://github.com/Lynx-Zhang/DD2424-Project.git'
REPO_DIR = '/content/DD2424-Project'
BRANCH = 'feat/sonnet-generation'

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}

print("Current branch: ", end='')
!git branch --show-current

!mkdir -p predictions
!mkdir -p /content/drive/MyDrive/sonnet_checkpoints
!mkdir -p /content/drive/MyDrive/sonnet_logs

print("\nGPU info:")
!nvidia-smi -L

print("\nEnvironment ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/DD2424-Project
Fetching origin
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 315.33 KiB | 5.09 MiB/s, done.
From https://github.com/Lynx-Zhang/DD2424-Project
   7fbc853..f4896e1  feat/sonnet-generation -> origin/feat/sonnet-generation
Already on 'feat/sonnet-generation'
Your branch is behind 'origin/feat/sonnet-generation' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Lynx-Zhang/DD2424-Project
 * branch            feat/sonnet-generation -> FETCH_HEAD
Updating 7fbc853..f4896e1
Fast-forward
 SonnetGeneration_Colab.ipynb | 21262 ++++++++++++++++++++++++++++++++++++++++-
 sonnet_generation.py         |   599 +

## 2. Install Dependencies

In [4]:
!pip install -q \
    tqdm==4.58.0 \
    requests==2.25.1 \
    importlib-metadata==3.7.0 \
    filelock==3.0.12 \
    tokenizers==0.20 \
    explainaboard_client==0.0.7 \
    einops==0.8.0 \
    transformers==4.46.3 \
    sacrebleu==2.5.1 \
    scikit-learn

print("Dependencies installed.")

Dependencies installed.


## 3. Verify Data Files

In [5]:
!echo "Training sonnets:"
!wc -l data/sonnets.txt

!echo "\nHeld-out sonnets (test, first 3 lines only):"
!wc -l data/sonnets_held_out.txt

!echo "\nHeld-out dev sonnets (val prompts, first 3 lines):"
!wc -l data/sonnets_held_out_dev.txt

!echo "\nTrue held-out dev sonnets (val references):"
!wc -l data/TRUE_sonnets_held_out_dev.txt

Training sonnets:
2237 data/sonnets.txt
\nHeld-out sonnets (test, first 3 lines only):
82 data/sonnets_held_out.txt
\nHeld-out dev sonnets (val prompts, first 3 lines):
81 data/sonnets_held_out_dev.txt
\nTrue held-out dev sonnets (val references):
213 data/TRUE_sonnets_held_out_dev.txt


## 4. Train — `gpt2` (small, fast baseline)

Trains with early stopping based on validation chrF score.  
The best checkpoint is saved to `best_<epochs>-<lr>-sonnet.pt`.

In [ ]:
!python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2 \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 8 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.9 \
    --sonnet_out predictions/generated_sonnets_gpt2.txt

# Backup checkpoint and predictions to Drive
!cp best_20-1e-05-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_gpt2-$(date +%Y%m%d_%H%M%S).pt
!cp predictions/generated_sonnets_gpt2.txt /content/drive/MyDrive/sonnet_logs/generated_sonnets_gpt2-$(date +%Y%m%d_%H%M%S).txt
print("\nBackup done.")

2026-05-12 13:51:30.206601: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
train-0: 100% 17/17 [00:08<00:00,  1.95it/s]
Epoch 0: train loss :: 4.918, val loss :: 4.327, val chrF :: 40.80.
save the model to best_20-1e-05-sonnet.pt
Generating several output sonnets...
Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake; every day exposed myself beneath void tears
And cowed by love like burn of darkness enveloped within it. Divine Lady, master of all my struggles, the beauty that we share in my smiling mind, I trample where erring realms fall; the blasphemous ethenage of natural hatred to dispel, I glory for what some might raise against me ; come to that sole limb who eateth acorn

^C
^C

Backup done.


## 5. (Optional) Train — `gpt2-medium` (better quality, slower)

Uses a larger model. Recommended only if you have a GPU with ≥ 15 GB VRAM (e.g., A100 on Colab Pro, barely running on T4).

In [13]:
!python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2-medium \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 4 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.9 \
    --sonnet_out predictions/generated_sonnets_gpt2medium.txt

!cp best_20-1e-05-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_gpt2medium-$(date +%Y%m%d_%H%M%S).pt
!cp predictions/generated_sonnets_gpt2medium.txt /content/drive/MyDrive/sonnet_logs/generated_sonnets_gpt2medium-$(date +%Y%m%d_%H%M%S).txt
print("\nBackup done.")

config.json: 100% 718/718 [00:00<00:00, 4.99MB/s]
model.safetensors: 100% 1.52G/1.52G [00:12<00:00, 118MB/s]
train-0: 100% 33/33 [00:27<00:00,  1.18it/s]
Epoch 0: train loss :: 4.453, val loss :: 3.981, val chrF :: 41.62.
save the model to best_20-1e-05-sonnet.pt
Generating several output sonnets...
Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake; This I burned to free thee, which stillseth My till the week comes,
God's way in perfect rapture you,
Now my bad fairy ruined and ashes chained in my shame.'' July 23rd 1849. Fanny's desire usurped over all virtues or knits, when kept up a maido bitch; still does wretched case now ring. . Eternal most love play', writing scorn properly did draw quick, of Come Love. Brothers [and sisters-in-love ad mss to xxiii] know music 'gainst me not by itself but for refreshment.' However I was just like Fairy to play


Poor soul, the center of my sinful earth,
Pressed with these rebe

## 6. Inspect Generated Sonnets

In [6]:
OUTPUT_FILE = 'predictions/generated_sonnets_gpt2.txt'  # change to gpt2medium if needed

with open(OUTPUT_FILE) as f:
    content = f.read()

print(content[:3000])

--Generated Sonnets-- 


0
Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake;
For the great strong worth, either divine me,
Which since sowed dead themas sweet pleasure,
Had there, beloved longer than Me, another term could sing?
Therefore did same knows Me rather in thine heart.
Even happier, therefore, than with Me this love of love,
Nor oft too cold nor dim, and we must say that faultless
Spotted blood held by no arts worse than Mine own;
  Be this my witness that I hated thy crime  And methinks no friend
  Less dead than slain ass on my score;
  I have blessed thee, though thee so


1
Poor soul, the center of my sinful earth,
Pressed with these rebel powers that thee array,
Why dost thou pine within and suffer dearth,
For desire'st not death prevent thy growing youth,
Yet used for a degree's estimate?
Much must I heartily love thee, that thou aliest sing!
Ask not thy freshness
To slight thy green habitation, thou

## 7. Re-generate from a Saved Checkpoint (without retraining)

Useful if training already finished and you want to regenerate with different sampling parameters.

In [ ]:
# Restore checkpoint from Drive if needed
# !cp /content/drive/MyDrive/sonnet_checkpoints/<your_checkpoint>.pt best_20-1e-05-sonnet.pt

import torch
import sys
sys.path.insert(0, '/content/DD2424-Project')

from sonnet_generation import SonnetGPT, generate_submission_sonnets, add_arguments
from datasets import SonnetsDataset
import argparse

# Mirror the args used during training
args = argparse.Namespace(
    use_gpu=True,
    model_size='gpt2',
    epochs=20,
    lr=1e-5,
    temperature=1.2,
    top_p=0.9,
    held_out_sonnet_path='data/sonnets_held_out.txt',
    sonnet_out='predictions/generated_sonnets_regen.txt',
)
args.filepath = f'{args.epochs}-{args.lr}-sonnet.pt'

generate_submission_sonnets(args)
print("\nRegeneration complete -> predictions/generated_sonnets_regen.txt")

## 8. Download Predictions

In [7]:
from google.colab import files
files.download('predictions/generated_sonnets_gpt2.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Generation hyperparameter tuning (temperature, top_p)

In [8]:
"""
Stage 1: Generation hyperparameter tuning (temperature, top_p)
No retraining needed, uses existing checkpoint.
"""
import torch
import sys
import os
import numpy as np
sys.path.insert(0, '/content/DD2424-Project')
os.chdir('/content/DD2424-Project')
from sonnet_generation import SonnetGPT, eval_chrf
from datasets import SonnetsDataset

# ========== Load best checkpoint ==========
CKPT_PATH = 'best_20-1e-05-sonnet.pt'
print(f"Loading checkpoint: {CKPT_PATH}")
saved = torch.load(CKPT_PATH, weights_only=False)
model = SonnetGPT(saved['args'])
model.load_state_dict(saved['model'])
model = model.to('cuda')
model.eval()

# ========== Load val set ==========
val_prompts = SonnetsDataset('data/sonnets_held_out_dev.txt')
val_refs = SonnetsDataset('data/TRUE_sonnets_held_out_dev.txt')
print(f"Val set size: {len(val_prompts)}")

# ========== Grid search ==========
configs = [
    # (temperature, top_p)
    (0.5, 0.9),    # near greedy
    (0.7, 0.9),    # conservative
    (0.9, 0.9),    # balanced
    (1.0, 0.9),    # neutral
    (1.2, 0.9),    # baseline
    (1.5, 0.9),    # high randomness

    (0.9, 0.80),
    (0.9, 0.85),
    (0.9, 0.95),
    (1.0, 0.85),
    (1.0, 0.95),
    (1.2, 0.85),
    (1.2, 0.95),
]

# Run each config N times to average out sampling randomness
N_RUNS = 3

print(f"\n{'='*60}")
print(f"Grid search ({N_RUNS} runs per config for stability)")
print(f"Baseline: T=1.2, top_p=0.9, chrF=42.39")
print(f"{'='*60}\n")

results = []
for temp, tp in configs:
    scores = []
    for run in range(N_RUNS):
        # Set different seeds to simulate multiple samples
        torch.manual_seed(11711 + run)
        np.random.seed(11711 + run)

        chrf = eval_chrf(model, val_prompts, val_refs, 'cuda',
                         temperature=temp, top_p=tp)
        scores.append(chrf)

    mean = np.mean(scores)
    std = np.std(scores)
    delta = mean - 42.39
    marker = " [BEST]" if mean > 42.39 else ""

    print(f"T={temp:.2f}, top_p={tp:.2f}: "
          f"chrF={mean:.2f} +/- {std:.2f} ({delta:+.2f}){marker}")
    results.append((temp, tp, mean, std))

# ========== Output best ==========
results.sort(key=lambda x: -x[2])
print(f"\n{'='*60}")
print("Top 5 configurations:")
print(f"{'='*60}")
for i, (t, p, m, s) in enumerate(results[:5]):
    print(f"  {i+1}. T={t}, top_p={p}: chrF={m:.2f} +/- {s:.2f}")

best_t, best_p, best_chrf, _ = results[0]
print(f"\n*** Best: T={best_t}, top_p={best_p} -> chrF={best_chrf:.2f} ***")
print(f"    Improvement over baseline: {best_chrf - 42.39:+.2f}")

Loading checkpoint: best_20-1e-05-sonnet.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Val set size: 12

Grid search (3 runs per config for stability)
Baseline: T=1.2, top_p=0.9, chrF=42.39

T=0.50, top_p=0.90: chrF=35.60 +/- 0.50 (-6.79)
T=0.70, top_p=0.90: chrF=40.14 +/- 0.40 (-2.25)
T=0.90, top_p=0.90: chrF=41.66 +/- 0.19 (-0.73)
T=1.00, top_p=0.90: chrF=41.85 +/- 0.11 (-0.54)
T=1.20, top_p=0.90: chrF=41.95 +/- 0.20 (-0.44)
T=1.50, top_p=0.90: chrF=40.80 +/- 0.32 (-1.59)
T=0.90, top_p=0.80: chrF=41.41 +/- 0.29 (-0.98)
T=0.90, top_p=0.85: chrF=41.72 +/- 0.48 (-0.67)
T=0.90, top_p=0.95: chrF=41.92 +/- 0.18 (-0.47)
T=1.00, top_p=0.85: chrF=42.09 +/- 0.08 (-0.30)
T=1.00, top_p=0.95: chrF=42.07 +/- 0.07 (-0.32)
T=1.20, top_p=0.85: chrF=42.13 +/- 0.39 (-0.26)
T=1.20, top_p=0.95: chrF=41.86 +/- 0.21 (-0.53)

Top 5 configurations:
  1. T=1.2, top_p=0.85: chrF=42.13 +/- 0.39
  2. T=1.0, top_p=0.85: chrF=42.09 +/- 0.08
  3. T=1.0, top_p=0.95: chrF=42.07 +/- 0.07
  4. T=1.2, top_p=0.9: chrF=41.95 +/- 0.20
  5. T=0.9, top_p=0.95: chrF=41.92 +/- 0.18

*** Best: T=1.2, top_p=0.85 -

with best temperature and $TOP_P$, next find other parameters


In [9]:
import datetime
import os

# 用阶段 1 找到的最佳生成参数
BEST_TEMP = 1.2
BEST_TOP_P = 0.85

experiments = [
    # (name, lr, epochs, batch_size, patience)
    ('A_lr1e5_e40',  '1e-5', 40, 8,  8),   # 更长训练
    ('B_lr3e5',      '3e-5', 25, 8,  5),   # 中等 lr
    ('C_lr5e5',      '5e-5', 15, 8,  5),   # 大 lr
    ('D_lr5e6',      '5e-6', 50, 8,  10),  # 小 lr + 长训练
    ('E_bs4',        '1e-5', 20, 4,  5),   # 小 batch
]

os.makedirs('/content/drive/MyDrive/sonnet_logs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/sonnet_checkpoints', exist_ok=True)

for name, lr, epochs, bs, patience in experiments:
    ts = datetime.datetime.now().strftime('%H%M%S')
    log_file = f'/content/drive/MyDrive/sonnet_logs/exp_{name}_{ts}.log'
    
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"  lr={lr}, epochs={epochs}, batch_size={bs}, patience={patience}")
    print(f"  T={BEST_TEMP}, top_p={BEST_TOP_P}")
    print(f"  Log: {log_file}")
    print(f"{'='*60}")
    
    !python -u sonnet_generation.py \
        --use_gpu \
        --model_size gpt2 \
        --epochs {epochs} \
        --lr {lr} \
        --batch_size {bs} \
        --patience {patience} \
        --temperature {BEST_TEMP} \
        --top_p {BEST_TOP_P} \
        --sonnet_out predictions/sonnets_{name}.txt 2>&1 | tee {log_file}
    
    # 备份 checkpoint（以防被覆盖）
    !cp best_{epochs}-{lr}-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_{name}.pt 2>/dev/null
    
    print(f"\nDone: {name}\n")

print("="*60)
print("All experiments complete!")
print("="*60)
!ls -lh /content/drive/MyDrive/sonnet_logs/exp_*


Experiment: A_lr1e5_e40
  lr=1e-5, epochs=40, batch_size=8, patience=8
  T=1.2, top_p=0.85
  Log: /content/drive/MyDrive/sonnet_logs/exp_A_lr1e5_e40_093554.log
train-0: 100%|██████████| 17/17 [00:08<00:00,  1.91it/s]
Epoch 0: train loss :: 4.918, val loss :: 4.327, val chrF :: 40.74.
save the model to best_40-1e-05-sonnet.pt
Generating several output sonnets...
Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake; but in day and night
Before love. Yet lo, gone tonight will never return. When death left; grieving of love: night were darkened is the life
The widow weeping in pines, smiling, and free, quiet with rose veil hidden. And my words be their exception. For noth Cannot her grief goun besides because of her love that had betrayed her words


Broken their sorrows of my last joy. The hope that bids should have been imparted by my praise. Fear proved which lest the paleness beneath attend. "Then first understand and 

For summerize

In [11]:
import re
import glob
import os

log_files = sorted(glob.glob('/content/drive/MyDrive/sonnet_logs/exp_*.log'))

# 提取每个实验的最佳状态
all_results = []

for log_path in log_files:
    name = os.path.basename(log_path).replace('exp_', '').rsplit('_', 1)[0]
    
    with open(log_path) as f:
        content = f.read()
    
    # 提取每一行的 train loss, val loss, val chrF
    pattern = r'Epoch (\d+): train loss :: (\d+\.\d+), val loss :: (\d+\.\d+), val chrF :: (\d+\.\d+)'
    epochs = re.findall(pattern, content)
    
    if not epochs:
        print(f"WARNING: no epoch data in {name}")
        continue
    
    # 找最佳 epoch（按 chrF）
    best_epoch = max(epochs, key=lambda e: float(e[3]))
    epoch_idx, train_loss, val_loss, chrf = best_epoch
    
    all_results.append({
        'name': name,
        'best_epoch': int(epoch_idx),
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'chrf': float(chrf),
        'total_epochs': len(epochs),
    })

# 加上 baseline
all_results.append({
    'name': 'baseline_e20',
    'best_epoch': '-',
    'train_loss': '-',
    'val_loss': '-',
    'chrf': 42.13,
    'total_epochs': '-',
})

# 排序
all_results.sort(key=lambda x: -x['chrf'])
max_chrf = max(r['chrf'] for r in all_results)

print("="*90)
print(f"{'Name':<25}{'Best Epoch':<12}{'Train Loss':<12}{'Val Loss':<12}{'chrF':<10}{'Total Ep':<10}")
print("="*90)
baseline_chrf = 42.13
for r in all_results:
    marker = " <-- BEST" if r['chrf'] == max_chrf else ""
    train_loss_str = f"{r['train_loss']:.3f}" if r['train_loss'] != '-' else '-'
    val_loss_str = f"{r['val_loss']:.3f}" if r['val_loss'] != '-' else '-'
    delta = r['chrf'] - baseline_chrf
    
    print(f"{r['name']:<25}{str(r['best_epoch']):<12}{train_loss_str:<12}"
          f"{val_loss_str:<12}{r['chrf']:>6.2f}    {str(r['total_epochs']):<10}{marker}")

Name                     Best Epoch  Train Loss  Val Loss    chrF      Total Ep  
B_lr3e5                  9           3.545       4.138        42.65    15         <-- BEST
C_lr5e5                  9           3.128       4.302        42.56    15        
D_lr5e6                  10          4.322       4.100        42.35    21        
A_lr1e5_e40              9           4.107       4.061        42.20    18        
E_bs4                    0           4.832       4.267        42.20    6         
baseline_e20             -           -           -            42.13    -         


Verify from the dataset again

After systematic hyperparameter tuning across both generation and training settings,
GPT-2 small achieves its best validation chrF score of **42.65** on the sonnet generation
task using the configuration described below. However, statistical analysis (see Section
"Discussion") suggests that GPT-2 small has reached its capacity ceiling around chrF ~42
on this dataset.

### Best Configuration

| Category | Parameter | Value |
|---|---|---|
| **Model** | model_size | `gpt2` (124M parameters) |
| | hidden_size (d) | 768 |
| | num_layers (l) | 12 |
| | num_heads | 12 |
| **Training** | learning_rate | `3e-5` |
| | batch_size | 8 |
| | epochs (best) | 9 (out of 15 trained) |
| | optimizer | AdamW (custom implementation) |
| | early_stopping_patience | 5 |
| **Generation** | temperature | 1.2 |
| | top_p (nucleus sampling) | 0.85 |
| | max_length | 128 |
| **Other** | random_seed | 11711 |
| | val set | 12 held-out sonnets (3-line prompt) |
| | val metric | chrF (sacrebleu) |

### Result

- **Best validation chrF: 42.65**
- Best epoch: 9 (early stopped at epoch 14 after 5 epochs without improvement)
- Train loss at best epoch: 3.545
- Val loss at best epoch: 4.138
- Improvement over baseline (T=1.2, top_p=0.9, lr=1e-5): +0.52 chrF

### Reproducing this Result

```bash
python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2 \
    --epochs 25 \
    --lr 3e-5 \
    --batch_size 8 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.85
```


Fine-tuning on max_length

In [16]:
# 看你最好的那次生成结果
!head -100 predictions/sonnets_B_lr3e5.txt

--Generated Sonnets-- 


0
Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake;
They would use
If they saw mine in their decease;
So let mine be told, as thou yourself do me,
From by thy love shall I write the true
Even when they hold their eye out from mine;
For thou art love's shadow upon every wrong in me,
Which lips doth believe tell you the truth.
  To write them I to a thousand faults imputed,
  Told that I knew thou no errant, or, to speak quite so,
  If they write to me, I swear thee now or then.
  Be assured that no being abused and


1
Poor soul, the center of my sinful earth,
Pressed with these rebel powers that thee array,
Why dost thou pine within and suffer dearth,
Who with thine eyes are smitten with thoughts like mine own,
And woe where thou shalt live?
He says I benefit nothing as much as the goddess,
Even for evermore the ocean and thine eye's seclusion.
Who durst, who with eyes this thou delight,
To 

In [ ]:
train with decay

In [ ]:
import datetime
import os

os.makedirs('/content/drive/MyDrive/sonnet_logs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/sonnet_checkpoints', exist_ok=True)

# 改 train() 函数加 weight_decay 参数后跑：
experiments = [
    # (name, lr, epochs, weight_decay)
    ('wd001', '3e-5', 15, '0.01'),
    ('wd01',  '3e-5', 15, '0.1'),
]

for name, lr, epochs, wd in experiments:
    ts = datetime.datetime.now().strftime('%H%M%S')
    log_file = f'/content/drive/MyDrive/sonnet_logs/exp_{name}_{ts}.log'
    
    !python -u sonnet_generation.py \
        --use_gpu --model_size gpt2 \
        --epochs {epochs} --lr {lr} --batch_size 8 \
        --temperature 1.2 --top_p 0.85 \
        --weight_decay {wd} \
        --sonnet_out predictions/sonnets_{name}.txt 2>&1 | tee {log_file}